# HVIA Trial Training Task — Olist Business & Data Analysis

**Goal:** investigate the Olist Brazilian E-Commerce Public Dataset from a business perspective, identify actionable patterns, and translate them into a realistic Data/AI opportunity for HVIA.

> This notebook is designed to be reproducible on the original 9 Olist CSV files. The public dataset covers 2016–2018, so the findings are historical and should not be described as a current 2026 operational snapshot.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA = "data/"

files = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

orders = pd.read_csv(DATA + files["orders"])
customers = pd.read_csv(DATA + files["customers"])
items = pd.read_csv(DATA + files["items"])
products = pd.read_csv(DATA + files["products"])
sellers = pd.read_csv(DATA + files["sellers"])
payments = pd.read_csv(DATA + files["payments"])
reviews = pd.read_csv(DATA + files["reviews"])
geolocation = pd.read_csv(DATA + files["geolocation"])
translation = pd.read_csv(DATA + files["translation"])


## 1. Dataset discovery

The dataset is a normalized relational model. `orders` is the central order table; it connects customers, order items, payments and reviews. Order items connect products and sellers. Product categories can be translated with the category lookup table.


In [ ]:
table_summary = pd.DataFrame({
    "table": ["orders","customers","items","products","sellers","payments","reviews","geolocation","translation"],
    "rows": [len(orders),len(customers),len(items),len(products),len(sellers),len(payments),len(reviews),len(geolocation),len(translation)]
})
table_summary


## 2. Data quality checks

Check duplicates, missing values and date fields before analysis. Do not blindly fill every missing value: missing delivery dates can represent orders that were not delivered, and that is a business state rather than a value to hide.


In [ ]:
quality = {}
for name, df in {
    "orders": orders, "customers": customers, "items": items, "products": products,
    "sellers": sellers, "payments": payments, "reviews": reviews
}.items():
    quality[name] = {
        "rows": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum())
    }

pd.DataFrame(quality).T


In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders["purchase_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders["delivery_delta_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

orders["delivered_late"] = (orders["delivery_delta_days"] > 0).astype("Int64")
orders["delivered_flag"] = orders["order_delivered_customer_date"].notna()


## 3. Build safe order-level aggregates

A common analytical mistake is joining one-to-many tables directly and then summing duplicated values. Aggregate payments, items and reviews to the order level first.


In [ ]:
items_order = (
    items.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        item_revenue=("price", "sum"),
        freight_revenue=("freight_value", "sum"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique")
    )
)

payments_order = (
    payments.groupby("order_id", as_index=False)
    .agg(
        payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count")
    )
)

reviews_order = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean")
    )

master = orders.merge(customers, on="customer_id", how="left")
master = master.merge(items_order, on="order_id", how="left")
master = master.merge(payments_order, on="order_id", how="left")
master = master.merge(reviews_order, on="order_id", how="left")

master.head()


## 4. Business KPI layer
The first KPI layer should answer:
- How large is the order base?
- How much commercial value is represented?
- What is the average order value?
- How often are orders late?
- How does delivery relate to satisfaction?


In [ ]:
kpis = {
    "orders": master["order_id"].nunique(),
    "customers": master["customer_unique_id"].nunique(),
    "item_plus_freight_revenue": master["item_revenue"].sum() + master["freight_revenue"].sum(),
    "avg_order_value_item_plus_freight": (master["item_revenue"] + master["freight_revenue"]).mean(),
    "avg_review_score": master["review_score"].mean(),
    "late_orders": int(master.loc[master["delivered_flag"] & master["delivered_late"].eq(1), "order_id"].nunique()),
}
kpis


## 5. Revenue and category analysis
Use item price plus freight for a transparent transactional revenue proxy. Payment value is kept as a separate measure because payment records can contain multiple rows per order.


In [ ]:
monthly = (
    master.dropna(subset=["purchase_month"])
    .groupby("purchase_month", as_index=False)
    .agg(
        revenue=("item_revenue","sum"),
        freight=("freight_revenue","sum"),
        orders=("order_id","nunique")
    )
)
monthly["total_value"] = monthly["revenue"] + monthly["freight"]

plt.figure(figsize=(12,5))
plt.plot(monthly["purchase_month"], monthly["total_value"])
plt.xticks(rotation=60)
plt.title("Monthly Item + Freight Value")
plt.xlabel("Purchase Month")
plt.ylabel("Value")
plt.tight_layout()
plt.show()


In [ ]:
product_enriched = items.merge(products[["product_id","product_category_name"]], on="product_id", how="left")
product_enriched = product_enriched.merge(translation, on="product_category_name", how="left")

category = (
    product_enriched.groupby("product_category_name_english", dropna=False)
    .agg(
        revenue=("price","sum"),
        orders=("order_id","nunique"),
        avg_item_price=("price","mean")
    )
    .sort_values("revenue", ascending=False)
)

category.head(10)


## 6. Geographic analysis
Compare revenue concentration and customer experience by state. Avoid interpreting geographic correlation as causation.


In [ ]:
geo = (
    master.groupby("customer_state", as_index=False)
    .agg(
        revenue=("item_revenue","sum"),
        orders=("order_id","nunique"),
        avg_review=("review_score","mean"),
        avg_delivery_days=("delivery_days","mean"),
        late_rate=("delivered_late","mean")
    )
    .sort_values("revenue", ascending=False)
)
geo.head(15)


## 7. Delivery and customer experience
This is the core business question:

**Is delivery performance associated with customer satisfaction?**

The public dataset analysis shows a large gap in average review score between on-time and late deliveries. This is an association, not proof of causality.


In [ ]:
delivery_review = (
    master[master["delivered_flag"]]
    .assign(
        delivery_status=lambda x: np.where(
            x["delivered_late"].eq(1), "Late", "On Time"
        )
    )
    .groupby("delivery_status", as_index=False)
    .agg(
        orders=("order_id","nunique"),
        avg_review=("review_score","mean"),
        avg_delivery_days=("delivery_days","mean")
    )
)
delivery_review


In [ ]:
plt.figure(figsize=(7,5))
sns.barplot(data=delivery_review, x="delivery_status", y="avg_review")
plt.title("Average Review Score by Delivery Status")
plt.xlabel("Delivery Status")
plt.ylabel("Average Review Score")
plt.tight_layout()
plt.show()


## 8. Seller performance diagnostic

Seller-level monitoring can identify concentration of operational problems. A production system should use minimum-volume thresholds before flagging a seller, so a seller with only one order is not treated like a high-volume seller.


In [ ]:
seller_order = (
    items.merge(orders[["order_id","customer_id","order_delivered_customer_date","order_estimated_delivery_date"]], on="order_id", how="left")
)

seller_order["late"] = (
    pd.to_datetime(seller_order["order_delivered_customer_date"], errors="coerce")
    > pd.to_datetime(seller_order["order_estimated_delivery_date"], errors="coerce")
)

seller_perf = (
    seller_order.groupby("seller_id", as_index=False)
    .agg(
        orders=("order_id","nunique"),
        revenue=("price","sum"),
        late_rate=("late","mean")
    )
)

seller_perf = seller_perf[seller_perf["orders"] >= 20].sort_values("late_rate", ascending=False)
seller_perf.head(10)


## 9. Business findings

The analysis should be communicated as findings, not as a list of charts.

1. **Delivery is commercially relevant:** the historical dataset shows a substantial association between late delivery and lower review scores.
2. **Revenue is geographically concentrated:** São Paulo is the largest revenue state and represents roughly 36% of revenue in one independent end-to-end analysis.
3. **Product mix is concentrated:** Health & Beauty is the leading revenue category in the cited analysis, followed by Watches & Gifts and Bed/Bath/Table.
4. **Payment behavior is concentrated:** credit card is the dominant payment method.
5. **Demand is seasonal:** November shows a strong revenue/order spike, consistent with Black Friday behavior.
6. **The dataset is operationally rich:** sellers, products, payments, reviews and delivery timestamps can be combined for monitoring and prediction.


## 10. HVIA solution concept — Delivery & Customer Experience Intelligence

### Phase 1 — Monitor
Power BI / web dashboard:
- on-time delivery rate
- average delivery days
- late orders
- review score
- seller and region breakdown
- revenue at risk

### Phase 2 — Diagnose
Use historical data to identify segments associated with late delivery:
- seller
- customer region
- product category
- basket characteristics
- purchase timing

### Phase 3 — Predict
Train a classification model to estimate late-delivery risk before the outcome is known.

### Phase 4 — Act
Generate alerts for high-risk orders and route them to operations/customer-service workflows.

### Phase 5 — Learn
Track model precision/recall, false alerts and business outcomes, then retrain periodically.


## 11. Important limitations

- The public dataset is historical (2016–2018).
- It does not contain a dedicated courier field; seller-level analysis is not a direct measure of courier performance.
- The dataset does not contain real-time traffic, weather, warehouse capacity or live tracking.
- Delivery/review relationships are associative; they do not establish causation.
- Current Olist has evolved significantly since the public dataset period, so a production engagement would require current operational data.


## 12. Final recommendation for the task submission

Do not present this as “I built a model and dashboard.” Present it as:

**Business question → evidence → interpretation → solution → measurable next step.**

The strongest story is the connection between operational delivery performance and customer experience, while revenue/geography/category analysis provides the commercial context.
